In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 20,
    "axes.labelsize": 40,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "legend.fontsize": 11,
    "figure.titlesize": 16,
})

GLOBAL_METHOD_COLORS = {
    "ConditionalShapley": "#ff7f0e",
    "Random": "#9467bd",
    "ann_approx_ConditionalShapley": "#d62728",
    "KNNShapley": "#1f77b4",
    "approx_ConditionalShapley": "#2ca02c",
}

GLOBAL_METHOD_MARKERS = {
    "ConditionalShapley": "o",
    "Random": "s",
    "ann_approx_ConditionalShapley": "^",
    "KNNShapley": "D",
    "approx_ConditionalShapley": "v",
}


GROUP_KEYS = [
    "dataset",
    "model",
    "n_train",
    "n_valid",
    "n_test",
    "b",
    "n_class",
    "train_shape",
    "valid_shape",
    "test_shape",
    "metric",
    "F",
]

def _norm_value(v):
    if isinstance(v, list):
        return tuple(_norm_value(x) for x in v)
    if isinstance(v, dict):
        return tuple((k, _norm_value(val)) for k, val in sorted(v.items()))
    return v

def _safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def _candidate_log_paths():
    cwd = Path.cwd()
    return [
        cwd / "logs" / "mylog.txt",
        cwd / "exps" / "logs" / "mylog.txt",
    ]

for _path in _candidate_log_paths():
    if _path.exists():
        LOG_PATH = _path
        break
else:
    raise FileNotFoundError("找不到 mylog.txt，请确认 notebook 的当前工作目录")

OUTPUT_DIR = LOG_PATH.parents[1] / "pics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using log: {LOG_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
def parse_log(path: Path):
    rows = []
    current_cfg = None

    for line in path.read_text().splitlines():
        start = line.find("{")
        if start < 0:
            continue
        obj = json.loads(line[start:])

        if "method" not in obj:
            current_cfg = obj
            continue

        if current_cfg is None:
            continue

        record = {k: current_cfg.get(k) for k in GROUP_KEYS + ["seed", "ver", "pythonhashseed"]}
        record["group_key"] = tuple(_norm_value(current_cfg.get(k)) for k in GROUP_KEYS)
        record.update({
            "method": obj["method"],
            "is_run": obj.get("is_run", False),
            "all_results": obj.get("all_results", {}),
        })
        rows.append(record)

    return rows

rows = parse_log(LOG_PATH)
df_rows = pd.DataFrame(rows)
df_rows["metric_name"] = df_rows["metric"]

print(f"parsed method records: {len(df_rows)}")
print(f"config groups: {df_rows['group_key'].nunique()}")
display(df_rows[["dataset", "model", "seed", "method"]].head())


In [ ]:
def build_curve_df(df):
    records = []
    for _, row in df.iterrows():
        results = row['all_results'] or {}
        axis = results.get('axis', [])
        metric_name = row['metric_name']
        y = results.get('add_high', []) if metric_name == 'accuracy' else results.get(metric_name, [])
        if len(axis) != len(y):
            continue
        for x, value in zip(axis, y):
            records.append({
                'group_key': row['group_key'],
                'seed': row['seed'],
                'method': row['method'],
                'axis': x,
                'value': float(value),
            })
    return pd.DataFrame(records)

df_curve = build_curve_df(df_rows)
if df_curve.empty:
    raise ValueError('没有解析到可绘制的曲线数据')

group_summary = (
    df_rows.groupby('group_key', as_index=False)
    .agg(seed_count=('seed', 'nunique'), method_count=('method', 'nunique'))
)
display(group_summary)
print('curve rows:', len(df_curve))
print('methods:', sorted(df_curve['method'].unique()))
print('seeds:', sorted(df_curve['seed'].dropna().unique()))

In [ ]:
def group_to_title(group_key):
    group = dict(zip(GROUP_KEYS, group_key))
    return f"{group['dataset']} | {group['model']} | train={group['n_train']} | shape={group['train_shape']} | b={group['b']} | seed-avg"

saved_files = []
for idx, group_key in enumerate(sorted(df_curve['group_key'].unique())):
    sub = df_curve[df_curve['group_key'] == group_key]
    agg = (
        sub.groupby(['method', 'axis'], as_index=False)
        .agg(mean=('value', 'mean'), std=('value', 'std'), count=('value', 'count'))
    )
    agg['std'] = agg['std'].fillna(0.0)

    order = (
        agg.sort_values(['method', 'axis'])
        .groupby('method', as_index=False)
        .tail(1)
        .sort_values('mean', ascending=False)['method']
        .tolist()
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    for method in order:
        m = agg[agg['method'] == method].sort_values('axis')
        x = m['axis'].to_numpy()
        mean = m['mean'].to_numpy()
        std = m['std'].to_numpy()
        color = GLOBAL_METHOD_COLORS.get(method)
        marker = GLOBAL_METHOD_MARKERS.get(method, 'o')
        markevery = max(1, len(x) // 20)
        ax.plot(x, mean, label=method, color=color, marker=marker, markevery=markevery, linewidth=2, markersize=12)
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.14, linewidth=0)

    group = dict(zip(GROUP_KEYS, group_key))
    ax.set_xlabel('number of selected points ($b$)')
    ax.set_ylabel(group['metric'])
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    name_parts = [
        _safe_name(group['dataset']),
        _safe_name(group['model']),
        f"train{group['n_train']}",
        f"b{group['b']}",
        f"shape{group['train_shape'][1]}",
        f"g{idx:02d}",
    ]
    fig_path = OUTPUT_DIR / ("_".join(name_parts) + '.pdf')
    fig.savefig(fig_path, bbox_inches='tight')
    saved_files.append(fig_path)
    plt.close(fig)

print('saved figures:')
for p in saved_files:
    print(p)
